In [5]:
import pandas as pd
import numpy as np

df = pd.read_csv("/content/HR Attrition.csv")
print(df.shape)
df.head()

(427, 11)


,emp_id,age_yrs,gender,department,job_title,commute_dist_km,base_salary_idr,satisfaction_score,attrition_status,commute_flag,salary_flag
0,EMP-8812,27.0,Male,Growth & Marketing,Content Strategist,22,9398606,1.0,Yes,Normal,Normal
1,EMP-5557,35.0,Male,Growth & Marketing,Social Media Specialist,24,8378117,3.0,Yes,Normal,Normal
2,EMP-3179,31.0,Female,Tech & Product,Product Manager,17,18170380,4.0,Yes,Normal,Normal
3,EMP-9275,28.0,Female,Operations,Supply Chain Admin,12,15931645,3.0,No,Normal,Normal
4,EMP-2422,27.0,Female,Tech & Product,Product Manager,32,18274615,4.0,Yes,Normal,Normal


In [7]:
def cek(nama, bermasalah, dimensi):
    return {"dimensi": dimensi, "aturan": nama,
            "jumlah_masalah": int(bermasalah.sum())}

hasil = [
    cek("Ada nilai kosong", df.isnull().any(axis=1), "Kelengkapan"),
    cek("Age di luar 18-65", ~df["age_yrs"].fillna(30).between(18, 65), "Keakuratan"),
    cek("Base salary <= 0", df["base_salary_idr"] <= 0, "Keakuratan"),
    cek("Employee ID duplikat", df["emp_id"].duplicated(keep=False), "Konsistensi"),
]

laporan = pd.DataFrame(hasil)
laporan["status"] = np.where(laporan["jumlah_masalah"] == 0, "Lulus", "Gagal")
laporan.loc[len(laporan)] = ["Ketepatan waktu", "Tidak ada kolom tanggal", None, "Tidak dapat diuji"]
laporan

,dimensi,aturan,jumlah_masalah,status
0,Kelengkapan,Ada nilai kosong,2,Gagal
1,Keakuratan,Age di luar 18-65,0,Lulus
2,Keakuratan,Base salary <= 0,0,Lulus
3,Konsistensi,Employee ID duplikat,18,Gagal
4,Ketepatan waktu,Tidak ada kolom tanggal,None,Tidak dapat diuji


In [9]:
import hashlib
import pandas as pd
import numpy as np

SALT = "ganti_dengan_rahasia_proyek"   # simpan terpisah dari dataset

def pseudonym(x):
    return "EMP_" + hashlib.sha256((SALT + str(x)).encode()).hexdigest()[:10]

df_masked = df.copy()
# Menggunakan 'emp_id' sebagai pengganti 'EmployeeNumber'
df_masked["emp_id"] = df["emp_id"].apply(pseudonym)

# Menggunakan 'age_yrs' sebagai pengganti 'Age'
df_masked["age_yrs"] = pd.cut(df["age_yrs"], [17, 25, 35, 45, 55, 100],
                             labels=["18-25", "26-35", "36-45", "46-55", "56+"])

# Menggunakan 'base_salary_idr' (dalam Rupiah) dengan rentang yang disesuaikan (misal dalam jutaan)
df_masked["base_salary_idr"] = pd.cut(df["base_salary_idr"],
                                       [0, 5000000, 10000000, 15000000, 20000000, np.inf],
                                       labels=["<5jt", "5-10jt", "10-15jt", "15-20jt", ">20jt"])

df_masked.to_csv("/content/HR_Attrition_masked.csv", index=False)
df_masked.head()

,emp_id,age_yrs,gender,department,job_title,commute_dist_km,base_salary_idr,satisfaction_score,attrition_status,commute_flag,salary_flag
0,EMP_d209583b1c,26-35,Male,Growth & Marketing,Content Strategist,22,5-10jt,1.0,Yes,Normal,Normal
1,EMP_ca829c443a,26-35,Male,Growth & Marketing,Social Media Specialist,24,5-10jt,3.0,Yes,Normal,Normal
2,EMP_2d880c5040,26-35,Female,Tech & Product,Product Manager,17,15-20jt,4.0,Yes,Normal,Normal
3,EMP_bdd3bc4e09,26-35,Female,Operations,Supply Chain Admin,12,15-20jt,3.0,No,Normal,Normal
4,EMP_09a32ea157,26-35,Female,Tech & Product,Product Manager,32,15-20jt,4.0,Yes,Normal,Normal


In [11]:
print(df_masked.groupby("department")["attrition_status"]
      .apply(lambda s: (s == "Yes").mean().round(3)))

department
Growth & Marketing    0.438
Operations            0.347
People & Culture      0.325
Tech & Product        0.339
Name: attrition_status, dtype: float64
